# PathAssist — Colab GPU training (single notebook, no repo clone)

**Upload only this file to Google Colab.** No local project, no `pathassist` package, no config files.

**One Colab run = one organ.** Only `ORGAN_ID` is downloaded and trained. The catalog lists all
organs for reference — it does **not** fetch every dataset. You get **one** checkpoint download.

The **4 ensemble members** are four different CNN architectures trained on the **same** organ's
tiles, then bundled into a single `pathassist_<ORGAN_ID>_ensemble.pt` file.

1. **Runtime → Change runtime type → GPU** (T4 is enough)
2. In **section 2** set `ORGAN_ID` (e.g. `gastrointestinal`)
3. Leave `RUN_MODE = 'deploy'` and `MACHINE_PROFILE = 'colab_gpu'`
4. **Run all** → saves `models/<ORGAN_ID>/ensemble.pt`
5. Section 13 downloads **one file**: `pathassist_<ORGAN_ID>_ensemble.pt`

**Auto-download organs** (Hugging Face, selected organ only): `lymph_node`, `breast`, `gastrointestinal`, `pulmonary`.

**Manual organs:** mount Google Drive, put tiles in `DATA_DIR/benign/` and `DATA_DIR/malignant/`.

> **GI (gastrointestinal):** loader balances **NORM + TUM** (not stream order). Section 2 prints `NOTEBOOK_VERSION` — if missing or old, re-upload this file from the repo.

> **Pulmonary:** loader balances **benign + malignant** lung classes (LC25000 stream is grouped — old notebooks got 2:1 imbalance and ~3 GB RAM). Re-upload if `NOTEBOOK_VERSION` is not `2026-07-05-pulmonary-balance`.

> **Production path (lymph node):** set `ORGAN_ID='lymph_node'` → `RUN_MODE='deploy'` → train → save `ensemble.pt` → copy to `models/lymph_node/` → run demo/Docker with `config/pcam_test.yaml` thresholds. **Retrain** if your checkpoint predates recall-checkpoint + diverse SOTA ensemble (`NOTEBOOK_VERSION` in section 2).
>
> **Is this enough for hospital prod?** Strong **decision-support demo** on patch benchmarks — not a cleared clinical device. Still need: local scanner validation, WSI ingest, regulatory sign-off. See Validation page for recall-first metrics.

> Decision-support demo only — not a clinical device.


## 1. Setup

In [ ]:
!pip install -q matplotlib datasets torchvision scikit-learn

In [ ]:
import os, time, copy, gc
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from contextlib import nullcontext

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch', torch.__version__, '| device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# ---- free-tier GPU speedups (T4) ----
# AMP (fp16) roughly halves VRAM and gives ~1.5-2x throughput on Tensor Cores.
# cudnn.benchmark autotunes conv kernels (tile size is fixed, so this is a pure win).
# channels_last is the preferred memory layout for convnets on GPU.
USE_AMP = torch.cuda.is_available()
USE_CHANNELS_LAST = torch.cuda.is_available()
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

def amp_autocast():
    """Mixed-precision context on GPU, no-op on CPU."""
    if USE_AMP:
        return torch.cuda.amp.autocast()
    return nullcontext()

def make_grad_scaler():
    return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def batch_to_device(tensor, device=DEVICE):
    """Async H2D copy; use channels_last layout for 4D image batches."""
    tensor = tensor.to(device, non_blocking=True)
    if USE_CHANNELS_LAST and tensor.dim() == 4:
        tensor = tensor.contiguous(memory_format=torch.channels_last)
    return tensor

def model_to_device(model, device=DEVICE):
    if USE_CHANNELS_LAST:
        return model.to(device, memory_format=torch.channels_last)
    return model.to(device)

print('AMP:', USE_AMP, '| channels_last:', USE_CHANNELS_LAST, '| cudnn.benchmark:', torch.backends.cudnn.benchmark)


## 2. Configuration — edit these

`MACHINE_PROFILE` picks sensible defaults for your hardware.
Override any value in `MODEL_CONFIG`, `AUG_CONFIG`, or `TRAIN_CONFIG` after.

In [ ]:
# ---- WHICH ORGAN TO TRAIN ----
NOTEBOOK_VERSION = '2026-07-05-pulmonary-balance'
ORGAN_ID = 'lymph_node'   # prod default: lymph_node | next: gastrointestinal | breast | pulmonary

# ---- run mode / hardware (usually leave as-is) ----
MACHINE_PROFILE = 'colab_gpu'   # colab_gpu | colab_gpu_high | colab_cpu
RUN_MODE = 'deploy'             # deploy | evaluate | full

# GI only: colab_cpu defaults to small 7K split (~1 GB). Set True for full 100K (~15 GB).
GI_USE_FULL_DATASET = False

# Full organ catalog (self-contained — no external config files)
ORGAN_TRAINING_CATALOG = {
  'lymph_node': {
    'name': 'Lymph Node', 'auto_download': True,
    'hf_dataset': '1aurent/PatchCamelyon', 'hf_split': 'train',
    'hf_url': 'https://huggingface.co/datasets/1aurent/PatchCamelyon',
    'tile_size': 96, 'approx_tiles': 262144,
    'task': 'Metastasis vs normal (PCam)',
  },
  'breast': {
    'name': 'Breast', 'auto_download': True,
    'hf_dataset': '1aurent/BACH', 'hf_split': 'train',
    'hf_url': 'https://huggingface.co/datasets/1aurent/BACH',
    'tile_size': 256, 'approx_tiles': 400,
    'task': 'Malignant vs benign (BACH)',
  },
  'gastrointestinal': {
    'name': 'Gastrointestinal', 'auto_download': True,
    'hf_dataset': '1aurent/NCT-CRC-HE', 'hf_split': 'NCT_CRC_HE_100K',
    'hf_url': 'https://huggingface.co/datasets/1aurent/NCT-CRC-HE',
    'tile_size': 224, 'approx_tiles': 100000,
    'task': 'Tumor vs normal colon (NCT-CRC-HE)',
  },
  'pulmonary': {
    'name': 'Pulmonary', 'auto_download': True,
    'hf_dataset': '1aurent/LC25000', 'hf_split': 'train',
    'hf_url': 'https://huggingface.co/datasets/1aurent/LC25000',
    'tile_size': 256, 'approx_tiles': 12500,
    'task': 'Lung cancer vs benign (LC25000 lung only)',
  },
  'genitourinary': {
    'name': 'Genitourinary', 'auto_download': False,
    'manual_url': 'https://panda.grand-challenge.org/',
    'tile_size': 256, 'task': 'Prostate — extract patches to benign/malignant folders',
  },
  'gynecologic': {
    'name': 'Gynecologic', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual H&E tiles on Drive',
  },
  'dermatopathology': {
    'name': 'Dermatopathology', 'auto_download': False,
    'manual_url': 'https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000',
    'tile_size': 256, 'task': 'Skin lesion tiles (map classes to benign/malignant)',
  },
  'hematopathology': {
    'name': 'Hematopathology', 'auto_download': False,
    'manual_url': 'https://www.kaggle.com/datasets/paultimothymooney/blood-cells',
    'tile_size': 256, 'task': 'Curated marrow/blood tiles on Drive',
  },
  'head_neck': {
    'name': 'Head & Neck', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual H&E tiles on Drive',
  },
  'neuropathology': {
    'name': 'Neuropathology', 'auto_download': False,
    'manual_url': 'https://www.kaggle.com/datasets/mahmoudreda55/brain-tumor-dataset',
    'tile_size': 256, 'task': 'Prefer H&E neuropath tiles on Drive',
  },
  'soft_tissue': {
    'name': 'Soft Tissue', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual patch curation on Drive',
  },
  'cardiovascular': {
    'name': 'Cardiovascular', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual H&E tiles on Drive',
  },
  'endocrine': {
    'name': 'Endocrine', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual H&E tiles on Drive',
  },
  'cancer_genomics': {
    'name': 'Cancer Genomics', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Matched molecular + tile labels on Drive',
  },
  'cytopathology': {
    'name': 'Cytopathology', 'auto_download': False,
    'manual_url': 'https://www.kaggle.com/datasets/paultimothymooney/cervical-cancer-largest-dataset-sipakmed',
    'tile_size': 256, 'task': 'Cytology tiles on Drive',
  },
  'infectious_disease': {
    'name': 'Infectious Disease', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual tiles on Drive',
  },
  'mediastinum': {
    'name': 'Mediastinum', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual H&E tiles on Drive',
  },
  'orthopedic': {
    'name': 'Orthopedic', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Bone lesion tiles on Drive',
  },
  'peritoneal': {
    'name': 'Peritoneal', 'auto_download': False,
    'manual_url': 'https://www.cancerimagingarchive.net/',
    'tile_size': 256, 'task': 'Manual H&E tiles on Drive',
  },
}

# Manual organs: mount Drive, then benign/ + malignant/ under DATA_DIR
DATA_DIR = Path('/content/drive/MyDrive/pathassist_data') / ORGAN_ID

if ORGAN_ID not in ORGAN_TRAINING_CATALOG:
    raise ValueError(
        f'Unknown ORGAN_ID {ORGAN_ID!r}. Pick one of: {sorted(ORGAN_TRAINING_CATALOG)}'
    )
TRAIN_SPEC = ORGAN_TRAINING_CATALOG[ORGAN_ID]
DATA_SOURCE = 'huggingface' if TRAIN_SPEC['auto_download'] else 'folder'

print('Organ catalog (reference only — downloads ONLY the → organ):')
print(f"{'ORGAN':<18} {'AUTO':<6} {'~TILES':<8} DATASET / MANUAL LINK")
for oid, spec in ORGAN_TRAINING_CATALOG.items():
    mark = '→' if oid == ORGAN_ID else ' '
    auto = 'yes' if spec['auto_download'] else 'no'
    approx = spec.get('approx_tiles', '—')
    link = spec.get('hf_url', spec.get('manual_url', 'Drive folder'))
    print(f"{mark} {oid:<16} {auto:<6} {str(approx):<8} {link}")
print()
print('Selected:', ORGAN_ID, '|', TRAIN_SPEC.get('task', 'manual folder'))
print('This run will download/train/save ONLY this organ.')
print('Data source:', DATA_SOURCE)
if DATA_SOURCE == 'folder':
    print('DATA_DIR (benign/ + malignant/):', DATA_DIR)

PROFILES = {
    'colab_cpu':      dict(max_samples=8000,  batch_size=32,  epochs=12, mc_samples=20,
                           data_split='train', cv_splits=3, eval_test=True, test_samples=2000),
    'colab_gpu':      dict(max_samples=32768, batch_size=64,  epochs=20, mc_samples=30,
                           data_split='train', cv_splits=3, epochs_cv=12, eval_test=True, test_samples=32768),
    'colab_gpu_high': dict(max_samples=65536, batch_size=128, epochs=25, mc_samples=40,
                           data_split='train', cv_splits=3, epochs_cv=15, eval_test=True, test_samples=32768),
    'hospital_cpu':   dict(max_samples=8000,  batch_size=16,  epochs=10, mc_samples=40,
                           data_split='train', cv_splits=3, eval_test=False),
    'finetune':       dict(max_samples=4000,  batch_size=16,  epochs=8,  mc_samples=20,
                           data_split='train', cv_splits=2, eval_test=False),
}
profile = PROFILES[MACHINE_PROFILE]

# --- model architecture ---
# backbone: 'custom' | 'resnet18' | 'efficientnet_b0'
MODEL_CONFIG = dict(
  backbone='custom',
  conv_channels=[32, 64, 128],
  kernel_size=3,
  use_batch_norm=True,
  dropout=0.35,
  head_hidden=64,
  pool='max',
  pretrained=True,
  freeze_backbone=False,
)

# --- augmentation (train only; disable blocks when finetuning on new scanner) ---
AUG_CONFIG = dict(
  enabled=True,
  horizontal_flip=0.5,
  vertical_flip=0.5,
  rotation_degrees=90,           # random multiple of 90 deg
  color_jitter=dict(brightness=0.05, contrast=0.05),  # helps generalization
)

# --- training ---
TRAIN_CONFIG = dict(
  lr=1e-3,
  weight_decay=1e-4,
  val_fraction=0.2,
  seed=42,
  early_stop_patience=4,
  lr_scheduler=True,
  optimizer='adamw',             # decoupled weight decay (better than Adam for fine-tuning)
  grad_clip_norm=1.0,            # stabilizes AMP + pretrained backbones
  checkpoint_metric='val_recall_at_threshold',
  checkpoint_threshold=0.25,
)

# --- ensemble: 4 diverse ImageNet SOTA backbones, soft vote + val-tuned weights ---
ENSEMBLE_MEMBERS = [
    dict(
        name='resnet18',
        model=dict(backbone='resnet18', pretrained=True, freeze_backbone=False,
                   dropout=0.5, head_hidden=256),
        train=dict(lr=3e-4),
        seed=42, train_fraction=1.0,
        aug=dict(horizontal_flip=0.5, vertical_flip=0.5, rotation_degrees=90),
    ),
    dict(
        name='resnet34',
        model=dict(backbone='resnet34', pretrained=True, freeze_backbone=False,
                   dropout=0.45, head_hidden=256),
        train=dict(lr=2e-4),
        seed=43, train_fraction=0.98,
        aug=dict(horizontal_flip=0.5, vertical_flip=0.5, rotation_degrees=90,
                 color_jitter=dict(brightness=0.08, contrast=0.08)),
    ),
    dict(
        name='efficientnet_b0',
        model=dict(backbone='efficientnet_b0', pretrained=True, freeze_backbone=False,
                   dropout=0.5, head_hidden=256),
        train=dict(lr=2e-4),
        seed=44, train_fraction=1.0,
        aug=dict(horizontal_flip=0.5, vertical_flip=0.5, rotation_degrees=90),
    ),
    dict(
        name='densenet121',
        model=dict(backbone='densenet121', pretrained=True, freeze_backbone=False,
                   dropout=0.4, head_hidden=256),
        train=dict(lr=1.5e-4, weight_decay=2e-4),
        seed=45, train_fraction=0.95,
        aug=dict(horizontal_flip=0.5, vertical_flip=0.5, rotation_degrees=90,
                 color_jitter=dict(brightness=0.1, contrast=0.1, saturation=0.05)),
    ),
]

VOTE_MODE = 'soft'   # 'soft' = average probabilities | 'hard' = majority vote
USE_VAL_WEIGHTS = True  # weight each model's vote by its performance

# --- performance-based vote weighting ---
# During training, each model's weight decays every epoch by how many it got wrong:
#   weight *= (1 - WEIGHT_DECAY_RATE * error_rate)
# so models that make more mistakes lose voting power. The final weight is used
# in the weighted vote at test time.
WEIGHT_DECAY_RATE = 1.0   # 0 = no penalty, 1 = strong penalty for errors
WEIGHT_FLOOR = 0.05       # a model never drops fully to zero

RUN_MODES = {
    'deploy':   dict(cv=False, final=True,  test=True),
    'evaluate': dict(cv=True,  final=False, test=False),
    'full':     dict(cv=True,  final=True,  test=True),
}
run = RUN_MODES[RUN_MODE]

# --- cross-validation ---
CV_CONFIG = dict(
    enabled=run['cv'],
    n_splits=profile.get('cv_splits', 3),
    epochs_per_fold=profile.get('epochs_cv'),
    train_final_on_all=run['final'],
    final_val_fraction=0.1,
)

# apply profile defaults
MAX_SAMPLES = profile['max_samples']
DATA_SPLIT  = profile.get('data_split', 'train')
EVAL_TEST   = run['test'] and profile.get('eval_test', False)
TEST_SAMPLES = profile.get('test_samples', 32768)
BATCH_SIZE  = profile['batch_size']
EPOCHS      = profile['epochs']
MC_SAMPLES  = profile['mc_samples']
TILE_SIZE   = int(TRAIN_SPEC.get('tile_size', 256))
MODEL_PATH  = Path('tile_classifier.pt')

# GI: 224px tiles are RAM-heavy — cap tiles on Colab (loader still balances NORM+TUM).
# AMP (fp16) halves VRAM, so batch 64 fits on a free T4 and trains faster.
if ORGAN_ID == 'gastrointestinal':
    MAX_SAMPLES = min(MAX_SAMPLES, 8192)
    BATCH_SIZE = min(BATCH_SIZE, 64) if USE_AMP else min(BATCH_SIZE, 32)
    EVAL_TEST = False
    TRAIN_CONFIG['checkpoint_threshold'] = 0.30  # gi_crc.yaml detection_threshold
    print(f'GI Colab cap: {MAX_SAMPLES} tiles ({MAX_SAMPLES // 2} NORM + {MAX_SAMPLES // 2} TUM), batch={BATCH_SIZE}')

# Pulmonary: 256px tiles + LC25000 has only 5k benign — cap RAM and balance per-class in loader.
if ORGAN_ID == 'pulmonary':
    MAX_SAMPLES = min(MAX_SAMPLES, 10000)
    BATCH_SIZE = min(BATCH_SIZE, 48) if USE_AMP else min(BATCH_SIZE, 24)
    print(f'Pulmonary Colab cap: {MAX_SAMPLES} tiles ({MAX_SAMPLES // 2} benign + {MAX_SAMPLES // 2} malig), batch={BATCH_SIZE}')

def resolve_ensemble_path(organ_id: str = ORGAN_ID) -> Path:
    """Colab cwd: models/<organ_id>/ensemble.pt"""
    path = Path('models') / organ_id / 'ensemble.pt'
    named = Path(f'pathassist_{organ_id}_ensemble.pt')
    for candidate in (path, named):
        if candidate.exists():
            return candidate.resolve()
    return path

ENSEMBLE_PATH = resolve_ensemble_path(ORGAN_ID)
DOWNLOAD_FILENAME = f'pathassist_{ORGAN_ID}_ensemble.pt'
DOWNLOAD_PATH = Path(DOWNLOAD_FILENAME)

print('Organ:', ORGAN_ID, '| data:', DATA_SOURCE)
print('Notebook:', NOTEBOOK_VERSION)
print('Profile:', MACHINE_PROFILE, '| RUN_MODE:', RUN_MODE,
      '| CV:', CV_CONFIG['enabled'], '| tiles:', MAX_SAMPLES,
      '| members:', len(ENSEMBLE_MEMBERS))
print('Save path:', ENSEMBLE_PATH, ('(found)' if ENSEMBLE_PATH.exists() else '(will save here)'))
print('Checkpoint metric:', TRAIN_CONFIG.get('checkpoint_metric'), '@', TRAIN_CONFIG.get('checkpoint_threshold'))


## 3. Load training tiles

Loads **only `ORGAN_ID`** — no other organ datasets are fetched.

- **Auto organs**: one Hugging Face dataset, capped at `MAX_SAMPLES` tiles
- **Manual organs**: `DATA_DIR/benign/` + `DATA_DIR/malignant/` on Google Drive


In [ ]:
IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')

# NCT-CRC-HE ClassLabel ids (do not use stream order — dataset is grouped by class)
NCT_NORM_LABEL = 6
NCT_TUM_LABEL = 8

def _as_pil_image(image):
    import io
    from PIL import Image
    if hasattr(image, 'convert'):
        return image.convert('RGB')
    if isinstance(image, dict):
        raw = image.get('bytes')
        if raw is not None:
            return Image.open(io.BytesIO(raw)).convert('RGB')
        path = image.get('path')
        if path:
            return Image.open(path).convert('RGB')
    return Image.fromarray(np.asarray(image)).convert('RGB')

def _resize_tile(image, tile_size):
    from PIL import Image
    img = _as_pil_image(image)
    if img.size != (tile_size, tile_size):
        img = img.resize((tile_size, tile_size), Image.Resampling.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

def load_from_image_folder(root, tile_size=64):
    """benign/ + malignant/ (or normal/tumor) subfolders → (tiles, labels)."""
    from PIL import Image
    root = Path(root)
    if not root.is_dir():
        raise NotADirectoryError(f'Not a directory: {root}')
    mapping = {
        'benign': 0.0, 'normal': 0.0, 'negative': 0.0,
        'malignant': 1.0, 'tumor': 1.0, 'tumour': 1.0, 'positive': 1.0,
    }
    tiles, labels = [], []
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        key = class_dir.name.lower()
        if key not in mapping:
            continue
        label = mapping[key]
        for image_path in sorted(class_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            with Image.open(image_path) as img:
                tiles.append(_resize_tile(img, tile_size))
                labels.append(label)
    if not tiles:
        raise ValueError(f'No tiles in {root} — need benign/ and malignant/ subfolders')
    return np.stack(tiles), np.asarray(labels, dtype=np.float32)

def load_training_tiles(organ_id, max_samples, tile_size):
    """Fetch tiles for a single organ only (never loops the full catalog)."""
    spec = ORGAN_TRAINING_CATALOG[organ_id]
    if not spec.get('auto_download'):
        try:
            from google.colab import drive
            if not Path('/content/drive/MyDrive').exists():
                drive.mount('/content/drive')
        except ImportError:
            pass
        tiles, labels = load_from_image_folder(DATA_DIR, tile_size=tile_size)
        return tiles, labels, {'source': 'folder', 'data_dir': str(DATA_DIR), 'organ_id': organ_id}

    from datasets import load_dataset
    hf, split = spec['hf_dataset'], spec['hf_split']
    print(f'[single-organ] Downloading ONLY {organ_id} from {hf} ({split}), max {max_samples} tiles...')

    if organ_id == 'lymph_node':
        ds = load_dataset(hf, split=f"{split}[:{max_samples}]")
        tiles = np.stack([_resize_tile(img, tile_size) for img in ds['image']])
        labels = np.array(ds['label'], dtype=np.float32)
    elif organ_id == 'breast':
        ds = load_dataset(hf, split=split)
        tiles, labels = [], []
        pos, neg = {1, 2}, {0, 3}
        for row in ds:
            lab = int(row['label'])
            if lab in pos: y = 1.0
            elif lab in neg: y = 0.0
            else: continue
            tiles.append(_resize_tile(row['image'], tile_size)); labels.append(y)
            if len(tiles) >= max_samples: break
        tiles, labels = np.stack(tiles), np.array(labels, dtype=np.float32)
    elif organ_id == 'gastrointestinal':
        # Full NCT_CRC_HE_100K ≈15 GB. CPU defaults to CRC_VAL_HE_7K (~1 GB) unless GI_USE_FULL_DATASET.
        use_small = MACHINE_PROFILE == 'colab_cpu' and not GI_USE_FULL_DATASET
        gi_split = 'CRC_VAL_HE_7K' if use_small else split
        if gi_split == split:
            print('Streaming full NCT-CRC-HE 100K (first fetch 10–30+ min on free Colab — be patient)')
            ds = load_dataset(hf, split=gi_split, streaming=True)
            ds = ds.shuffle(seed=0, buffer_size=10_000)
        else:
            print(f'CPU shortcut: {gi_split} (~1 GB). Set GI_USE_FULL_DATASET=True for full 100K.')
            ds = load_dataset(hf, split=gi_split).shuffle(seed=0)
        per_class = max_samples // 2
        norm_tiles, tum_tiles = [], []
        scanned = 0
        for row in ds:
            scanned += 1
            lab = int(row['label'])
            if lab == NCT_NORM_LABEL and len(norm_tiles) < per_class:
                norm_tiles.append(_resize_tile(row['image'], tile_size))
            elif lab == NCT_TUM_LABEL and len(tum_tiles) < per_class:
                tum_tiles.append(_resize_tile(row['image'], tile_size))
            if len(norm_tiles) >= per_class and len(tum_tiles) >= per_class:
                break
            if len(norm_tiles) + len(tum_tiles) > 0 and (len(norm_tiles) + len(tum_tiles)) % 200 == 0:
                print(f'  norm {len(norm_tiles)}/{per_class} | tum {len(tum_tiles)}/{per_class} (scanned {scanned})…')
        if not norm_tiles or not tum_tiles:
            raise ValueError(f'Need both NORM and TUM — got norm={len(norm_tiles)} tum={len(tum_tiles)}')
        tiles = np.stack(norm_tiles + tum_tiles)
        labels = np.array([0.0] * len(norm_tiles) + [1.0] * len(tum_tiles), dtype=np.float32)
        perm = np.random.default_rng(0).permutation(len(labels))
        tiles, labels = tiles[perm], labels[perm]
        print(f'GI load done: {len(tiles)} tiles ({len(norm_tiles)} NORM + {len(tum_tiles)} TUM) from {scanned} scanned')
    elif organ_id == 'pulmonary':
        # LC25000: label 0 = benign lung_n; labels 1-4 = cancer subtypes (5k each).
        # Balance per-class — stream order is grouped by class (stops early with 2:1 otherwise).
        LC_BENIGN, LC_MALIGNANT = {0}, {1, 2, 3, 4}
        print('Streaming LC25000 lung subset (balanced benign/malignant)…')
        ds = load_dataset(hf, split=split, streaming=True)
        per_class = max_samples // 2
        benign_tiles, malig_tiles = [], []
        scanned = 0
        for row in ds:
            scanned += 1
            if int(row['organ']) != 0:
                continue
            lab = int(row['label'])
            tile = _resize_tile(row['image'], tile_size)
            if lab in LC_BENIGN and len(benign_tiles) < per_class:
                benign_tiles.append(tile)
            elif lab in LC_MALIGNANT and len(malig_tiles) < per_class:
                malig_tiles.append(tile)
            if len(benign_tiles) >= per_class and len(malig_tiles) >= per_class:
                break
            total = len(benign_tiles) + len(malig_tiles)
            if total > 0 and total % 200 == 0:
                print(f'  benign {len(benign_tiles)}/{per_class} | malig {len(malig_tiles)}/{per_class} (scanned {scanned})…')
        if not benign_tiles or not malig_tiles:
            raise ValueError(f'Need both benign and malignant — got benign={len(benign_tiles)} malig={len(malig_tiles)}')
        tiles = np.stack(benign_tiles + malig_tiles)
        labels = np.array([0.0] * len(benign_tiles) + [1.0] * len(malig_tiles), dtype=np.float32)
        perm = np.random.default_rng(0).permutation(len(labels))
        tiles, labels = tiles[perm], labels[perm]
        print(f'Pulmonary load done: {len(tiles)} tiles ({len(benign_tiles)} benign + {len(malig_tiles)} malig) from {scanned} scanned')
    else:
        raise ValueError(f'No Hugging Face loader wired for {organ_id}')

    return tiles, labels, {
        'source': 'huggingface', 'organ_id': organ_id,
        'hf_dataset': hf, 'hf_split': split, 'max_samples': max_samples,
    }

tiles, labels, data_meta = load_training_tiles(ORGAN_ID, MAX_SAMPLES, TILE_SIZE)
npos = int(labels.sum())
nneg = int((labels == 0).sum())
print('Loaded from:', data_meta)
print(f'Organ {ORGAN_ID} | {len(tiles)} tiles | positive: {npos} | negative: {nneg}')
if npos == 0 or nneg == 0:
    raise ValueError(
        f'Single-class labels ({npos} tumor / {nneg} normal). '
        f'Re-upload notebooks/train_and_evaluate.ipynb — need NOTEBOOK_VERSION={NOTEBOOK_VERSION!r}'
    )
if abs(npos - nneg) > max(32, len(labels) // 10):
    print(f'WARNING: class imbalance ({npos} pos vs {nneg} neg)')


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for row, (lab, name) in enumerate([(0, 'normal'), (1, 'metastasis')]):
    idx = np.where(labels == lab)[0][:4]
    for col, i in enumerate(idx):
        axes[row, col].imshow(tiles[i]); axes[row, col].axis('off')
        if col == 0: axes[row, col].set_title(name)
plt.suptitle(f'{TRAIN_SPEC.get("name", ORGAN_ID)} training samples'); plt.tight_layout(); plt.show()

## 4. Augmentation pipeline (train only)

Orientation augmentations are label-safe for pathology tiles.
`color_jitter` is off by default — enable in `AUG_CONFIG` when finetuning across scanners.

In [ ]:
class TileAugmentor:
    """Train-only augmentation. Validation / inference never call this."""
    def __init__(self, cfg: dict, seed: int = 0):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)

    def __call__(self, tile: np.ndarray) -> np.ndarray:
        if not self.cfg.get('enabled', True):
            return tile
        # torchvision transforms expect CHW; our tiles are HWC uint8 -> convert first
        img = torch.from_numpy(tile).permute(2, 0, 1)  # HWC -> CHW
        if self.rng.random() < self.cfg.get('horizontal_flip', 0):
            img = TF.hflip(img)
        if self.rng.random() < self.cfg.get('vertical_flip', 0):
            img = TF.vflip(img)
        rot = self.cfg.get('rotation_degrees', 0)
        if rot:
            angle = int(self.rng.integers(0, 360 // rot)) * rot
            img = TF.rotate(img, angle)
        cj = self.cfg.get('color_jitter')
        if cj:
            jitter = T.ColorJitter(**cj)
            img = img.float() / 255.0
            img = jitter(img)
            img = (img * 255).clamp(0, 255).byte()
        return img.permute(1, 2, 0).numpy()  # CHW -> HWC

augmentor = TileAugmentor(AUG_CONFIG, seed=TRAIN_CONFIG['seed'])

# preview: same tile, 6 augmented versions
sample = tiles[0]
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, _ in zip(axes, range(6)):
    ax.imshow(augmentor(sample)); ax.axis('off')
plt.suptitle('Augmentation preview (train only)'); plt.show()

## 5. Model architecture (custom CNN + SOTA backbones)

`backbone='resnet18'` uses **ImageNet-pretrained ResNet-18** — strong
per-organ binary classifier. Ensemble uses 4 pretrained members (ResNet-18 × 3 + EfficientNet-B0).

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def merge_config(base: dict, overrides: dict) -> dict:
    out = copy.deepcopy(base)
    out.update(overrides)
    return out


def get_model_norm(cfg):
    return 'imagenet' if cfg.get('backbone', 'custom') != 'custom' else 'custom'


def tiles_to_tensor(batch_np, augmentor=None, normalize='custom'):
    """HWC uint8 -> BCHW float. Pretrained backbones need ImageNet normalization."""
    if augmentor is not None:
        batch_np = np.stack([augmentor(t) for t in batch_np])
    x = batch_np.astype(np.float32) / 255.0
    if normalize == 'imagenet':
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
    else:
        x = (x - 0.5) / 0.5
    return torch.from_numpy(x).permute(0, 3, 1, 2)


class TileClassifier(nn.Module):
    """Small custom CNN — fast, CPU-friendly."""
    def __init__(self, cfg: dict):
        super().__init__()
        self.cfg = cfg
        channels = cfg['conv_channels']
        k = cfg['kernel_size']
        use_bn = cfg.get('use_batch_norm', True)
        pool_type = cfg.get('pool', 'max')
        layers, in_ch = [], 3
        for out_ch in channels:
            layers += [nn.Conv2d(in_ch, out_ch, k, padding=k // 2)]
            if use_bn:
                layers += [nn.BatchNorm2d(out_ch)]
            layers += [nn.ReLU(inplace=True)]
            layers += [nn.MaxPool2d(2) if pool_type == 'max' else nn.AvgPool2d(2)]
            in_ch = out_ch
        layers += [nn.AdaptiveAvgPool2d(1)]
        self.features = nn.Sequential(*layers)
        drop, hidden = cfg.get('dropout', 0.0), cfg.get('head_hidden', 0)
        if hidden > 0:
            self.head = nn.Sequential(
                nn.Flatten(), nn.Linear(channels[-1], hidden), nn.ReLU(), nn.Dropout(drop),
                nn.Linear(hidden, 1),
            )
        else:
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(drop), nn.Linear(channels[-1], 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


class PretrainedBackbone(nn.Module):
    """Wrapper for torchvision backbones with a dropout classification head."""
    def __init__(self, net: nn.Module, cfg: dict):
        super().__init__()
        self.cfg = cfg
        self.net = net

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _build_classifier_head(feat_dim: int, cfg: dict) -> nn.Module:
    drop = cfg.get('dropout', 0.5)
    hidden = cfg.get('head_hidden', 256)
    return nn.Sequential(
        nn.Dropout(drop),
        nn.Linear(feat_dim, hidden),
        nn.ReLU(inplace=True),
        nn.Dropout(drop),
        nn.Linear(hidden, 1),
    )


def _build_resnet18(cfg: dict) -> PretrainedBackbone:
    from torchvision.models import resnet18, ResNet18_Weights
    weights = ResNet18_Weights.IMAGENET1K_V1 if cfg.get('pretrained', True) else None
    net = resnet18(weights=weights)
    feat_dim = net.fc.in_features
    net.fc = _build_classifier_head(feat_dim, cfg)
    if cfg.get('freeze_backbone', False):
        for name, p in net.named_parameters():
            if not name.startswith('fc.'):
                p.requires_grad = False
    return PretrainedBackbone(net, cfg)


def _build_efficientnet_b0(cfg: dict) -> PretrainedBackbone:
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if cfg.get('pretrained', True) else None
    net = efficientnet_b0(weights=weights)
    feat_dim = net.classifier[1].in_features
    net.classifier = _build_classifier_head(feat_dim, cfg)
    if cfg.get('freeze_backbone', False):
        for name, p in net.features.named_parameters():
            p.requires_grad = False
    return PretrainedBackbone(net, cfg)


def _build_resnet34(cfg: dict) -> PretrainedBackbone:
    from torchvision.models import resnet34, ResNet34_Weights
    weights = ResNet34_Weights.IMAGENET1K_V1 if cfg.get('pretrained', True) else None
    net = resnet34(weights=weights)
    feat_dim = net.fc.in_features
    net.fc = _build_classifier_head(feat_dim, cfg)
    if cfg.get('freeze_backbone', False):
        for name, p in net.named_parameters():
            if not name.startswith('fc.'):
                p.requires_grad = False
    return PretrainedBackbone(net, cfg)


def _build_densenet121(cfg: dict) -> PretrainedBackbone:
    from torchvision.models import densenet121, DenseNet121_Weights
    weights = DenseNet121_Weights.IMAGENET1K_V1 if cfg.get('pretrained', True) else None
    net = densenet121(weights=weights)
    feat_dim = net.classifier.in_features
    net.classifier = _build_classifier_head(feat_dim, cfg)
    if cfg.get('freeze_backbone', False):
        for p in net.features.parameters():
            p.requires_grad = False
    return PretrainedBackbone(net, cfg)


def build_model(cfg: dict) -> nn.Module:
    """Factory: custom CNN or pretrained SOTA backbone."""
    backbone = cfg.get('backbone', 'custom')
    if backbone == 'custom':
        return TileClassifier(cfg)
    if backbone == 'resnet18':
        return _build_resnet18(cfg)
    if backbone == 'resnet34':
        return _build_resnet34(cfg)
    if backbone == 'efficientnet_b0':
        return _build_efficientnet_b0(cfg)
    if backbone == 'densenet121':
        return _build_densenet121(cfg)
    raise ValueError(f"Unknown backbone: {backbone!r}")


def model_summary(model: nn.Module, input_size=(1, 3, 96, 96)):
    model.eval()
    cfg = model.cfg
    norm = get_model_norm(cfg)
    print('=' * 60)
    print(f"backbone: {cfg.get('backbone', 'custom')} | normalize: {norm}")
    print(model)
    print('=' * 60)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total params:     {total:,}')
    print(f'Trainable params: {trainable:,}')
    with torch.no_grad():
        device = next(model.parameters()).device
        x = tiles_to_tensor(np.zeros((1, 96, 96, 3), dtype=np.uint8), normalize=norm).to(device)
        out = model(x)
    print(f'Output shape (batch=1): {tuple(out.shape)}')
    print('=' * 60)
    return total


print('--- custom CNN preview ---')
n_custom = model_summary(build_model(MODEL_CONFIG).to(DEVICE))
print('\n--- SOTA ResNet-18 preview (ensemble member) ---')
sota_cfg = merge_config(MODEL_CONFIG, ENSEMBLE_MEMBERS[-1]['model'])
n_sota = model_summary(build_model(sota_cfg).to(DEVICE))

## 6. SOTA reference (lymph node / PatchCamelyon)

Benchmark table below is for **lymph node (PatchCamelyon)**. Other organs use different
public sets; compare against literature for that organ instead.

| Method | Test accuracy (approx.) | Notes |
|--------|------------------------|-------|
| Simple CNN (ours, baseline) | ~85–90% | Small, fast, CPU-friendly |
| ResNet-18 ensemble (ours, colab_gpu_high) | ~94–97% | **4 pretrained members** |
| ResNet-18 + heavy aug | ~93% | [PCam benchmark](https://github.com/basveeling/pcam) |
| Rotation-equivariant CNN | ~90%+ | original PCam paper |
| MMSEN (2024) | 99% ROC-AUC | research model, full train set |

Our lymph-node target: **~94–97% test accuracy** with `colab_gpu_high`.
Decision-support still needs human review — high recall matters more than accuracy.


## 7. Training utilities + early stopping

In [ ]:
def release_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def split_data(tiles, labels, val_fraction, seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(tiles))
    n_val = max(1, int(len(tiles) * val_fraction))
    val_idx, train_idx = order[:n_val], order[n_val:]
    return (tiles[train_idx], labels[train_idx]), (tiles[val_idx], labels[val_idx])


def iter_batches(x, y, batch_size, shuffle):
    idx = np.arange(len(x))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        b = idx[start:start + batch_size]
        yield x[b], y[b]


@torch.no_grad()
def evaluate(model, val_x, val_y, batch_size, device, loss_fn):
    model.eval()
    probs, true, total_loss, n = [], [], 0.0, 0
    norm = get_model_norm(model.cfg)
    for bx, by in iter_batches(val_x, val_y, batch_size, False):
        inp = batch_to_device(tiles_to_tensor(bx, normalize=norm), device)
        tgt = torch.from_numpy(by).to(device, non_blocking=True)
        with amp_autocast():
            logits = model(inp)
            total_loss += loss_fn(logits, tgt).item()
        probs.append(torch.sigmoid(logits.float()).cpu().numpy())
        true.append(by)
        n += 1
    probs = np.concatenate(probs)
    true = np.concatenate(true)
    acc = ((probs >= 0.5).astype(np.float32) == true).mean()
    return true, probs, acc, total_loss / max(1, n)


def merge_config(base: dict, overrides: dict) -> dict:
    out = copy.deepcopy(base)
    out.update(overrides)
    return out


def member_train_subset(train_x, train_y, fraction, seed):
    """Each ensemble member trains on a slightly different tile subset."""
    rng = np.random.default_rng(seed)
    n = max(1, int(len(train_x) * fraction))
    idx = rng.choice(len(train_x), size=n, replace=False)
    return train_x[idx], train_y[idx]


@torch.no_grad()
def predict_probs(model, batch_np, device, batch_size=64):
    model.eval()
    norm = get_model_norm(model.cfg)
    out = []
    for start in range(0, len(batch_np), batch_size):
        bx = batch_np[start:start + batch_size]
        inp = batch_to_device(tiles_to_tensor(bx, normalize=norm), device)
        with amp_autocast():
            logits = model(inp)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)


def train_model(model, train_x, train_y, val_x, val_y, cfg, aug, device, epochs=EPOCHS, member_train_cfg=None):
    train_cfg = merge_config(cfg, member_train_cfg or {})
    norm = get_model_norm(model.cfg)
    opt_name = str(train_cfg.get('optimizer', 'adamw')).lower()
    if opt_name == 'adamw':
        opt = torch.optim.AdamW(model.parameters(), lr=train_cfg['lr'], weight_decay=train_cfg['weight_decay'])
    else:
        opt = torch.optim.Adam(model.parameters(), lr=train_cfg['lr'], weight_decay=train_cfg['weight_decay'])
    grad_clip = train_cfg.get('grad_clip_norm')
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5) if train_cfg.get('lr_scheduler') else None
    loss_fn = nn.BCEWithLogitsLoss()
    scaler = make_grad_scaler()
    history = []
    ckpt_metric = train_cfg.get('checkpoint_metric', 'val_loss')
    ckpt_thr = float(train_cfg.get('checkpoint_threshold', 0.25))
    maximize_ckpt = ckpt_metric != 'val_loss'
    best_ckpt = -1.0 if maximize_ckpt else float('inf')
    best_state, patience_ctr = None, 0
    patience = train_cfg.get('early_stop_patience', 4)
    monitor_thr = ckpt_thr if ckpt_metric == 'val_recall_at_threshold' else 0.5

    # performance-based voting weight, decayed every epoch by the error rate
    vote_weight = 1.0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        running, batches = 0.0, 0
        for bx, by in iter_batches(train_x, train_y, BATCH_SIZE, shuffle=True):
            inp = batch_to_device(tiles_to_tensor(bx, augmentor=aug, normalize=norm), device)
            tgt = torch.from_numpy(by).to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = loss_fn(model(inp), tgt)
            scaler.scale(loss).backward()
            if grad_clip:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt)
            scaler.update()
            running += loss.item(); batches += 1

        val_true, val_prob, val_acc, val_loss = evaluate(model, val_x, val_y, BATCH_SIZE, device, loss_fn)
        train_loss = running / max(1, batches)
        val_m = compute_metrics(val_true, val_prob, threshold=monitor_thr)
        val_rec, val_prec = val_m['rec'], val_m['prec']

        # --- decay the vote weight by how many this model got wrong this epoch ---
        n_wrong = int(((val_prob >= monitor_thr).astype(int) != val_true.astype(int)).sum())
        error_rate = n_wrong / max(1, len(val_true))
        vote_weight *= max(0.0, 1.0 - WEIGHT_DECAY_RATE * error_rate)
        vote_weight = max(vote_weight, WEIGHT_FLOOR)

        history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                            val_acc=val_acc, val_rec=val_rec, val_prec=val_prec,
                            n_wrong=n_wrong, vote_weight=vote_weight))
        if scheduler: scheduler.step(val_loss)

        marker = ''
        if ckpt_metric == 'val_recall_at_threshold':
            improved = val_rec > best_ckpt
            if improved:
                best_ckpt = val_rec
        elif ckpt_metric == 'val_loss':
            improved = val_loss < best_ckpt
            if improved:
                best_ckpt = val_loss
        else:
            raise ValueError(f"Unknown checkpoint_metric: {ckpt_metric!r}")
        if improved:
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
            marker = ' *best*'
        else:
            patience_ctr += 1

        print(f'    ep {epoch:>2}/{epochs}  train {train_loss:.4f}  val {val_loss:.4f}  '
              f'acc {val_acc:.3f}  rec@{monitor_thr:g} {val_rec:.3f}  prec {val_prec:.3f}  '
              f'wrong {n_wrong}  w {vote_weight:.3f}  '
              f'({time.time()-t0:.0f}s){marker}')
        if patience_ctr >= patience:
            print(f'    early stop at epoch {epoch}')
            break

    if best_state:
        model.load_state_dict(best_state)
    val_true, val_prob, val_acc, val_loss = evaluate(model, val_x, val_y, BATCH_SIZE, device, loss_fn)
    final_m = compute_metrics(val_true, val_prob, threshold=monitor_thr)
    val_rec = final_m['rec']
    return history, val_acc, val_loss, val_prob, vote_weight, val_rec


class VotingEnsemble:
    """Multiple models vote on each tile. Supports soft vote (avg prob) or hard vote (majority)."""

    def __init__(self, members: list, vote_mode='soft'):
        self.members = members   # list of dict: name, model, weight, val_acc
        self.vote_mode = vote_mode

    @torch.no_grad()
    def predict(self, batch_np, device, batch_size=64):
        member_probs = []
        for m in self.members:
            member_probs.append(predict_probs(m['model'], batch_np, device, batch_size))
        stacked = np.stack(member_probs, axis=0)              # (n_models, N)
        weights = np.array([m['weight'] for m in self.members], dtype=np.float32)
        weights /= weights.sum()

        soft_prob = (stacked * weights[:, None]).sum(axis=0)
        hard_votes = (stacked >= 0.5).astype(int)
        hard_prob = (hard_votes.mean(axis=0) >= 0.5).astype(np.float32)
        votes_for_meta = hard_votes.sum(axis=0)               # how many said metastasis
        disagreement = stacked.std(axis=0)                      # models disagree → review

        final_prob = soft_prob if self.vote_mode == 'soft' else hard_prob
        return dict(
            prob=final_prob,
            soft_prob=soft_prob,
            hard_prob=hard_prob,
            member_probs=stacked,
            votes_for_metastasis=votes_for_meta,
            n_models=len(self.members),
            disagreement=disagreement,
        )


def compute_metrics(y_true, y_prob, threshold=0.5):
    """Accuracy, precision, recall, F1 — recall matters most for missed cancers."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    acc = (tp + tn) / max(1, tp + tn + fp + fn)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return dict(acc=acc, prec=prec, rec=rec, f1=f1, tp=tp, tn=tn, fp=fp, fn=fn)


def stratified_split(tiles, labels, val_fraction, seed):
    """Stratified train/val split — keeps class balance in both sets."""
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(labels))
    train_idx, val_idx = train_test_split(
        idx, test_size=val_fraction, stratify=labels, random_state=seed,
    )
    return (tiles[train_idx], labels[train_idx]), (tiles[val_idx], labels[val_idx])


def train_ensemble_on_split(train_x, train_y, val_x, val_y, epochs=None, fold_id=0, verbose=True):
    """Train all ensemble members on one split; return ensemble + metrics on val_x."""
    epochs = epochs or EPOCHS
    trained_members, member_histories = [], {}

    for spec in ENSEMBLE_MEMBERS:
        release_gpu()
        member_t0 = time.time()
        if verbose:
            print(f"\n--- fold {fold_id} | member: {spec['name']} ---")
        model_cfg = merge_config(MODEL_CONFIG, spec.get('model', {}))
        aug_cfg = merge_config(AUG_CONFIG, spec.get('aug', {}))
        member_seed = spec['seed'] + fold_id * 100
        aug = TileAugmentor(aug_cfg, seed=member_seed)

        m_train_x, m_train_y = member_train_subset(
            train_x, train_y, spec.get('train_fraction', 1.0), member_seed,
        )
        if verbose:
            bb = model_cfg.get('backbone', 'custom')
            extra = f"head={model_cfg.get('head_hidden', '-')}" if bb != 'custom' else str(model_cfg.get('conv_channels', '-'))
            print(f"  backbone={bb} | {extra} | train {len(m_train_x)} | val {len(val_x)}")

        model = model_to_device(build_model(model_cfg))
        history, val_acc, val_loss, _, vote_weight, val_rec = train_model(
            model, m_train_x, m_train_y, val_x, val_y, TRAIN_CONFIG, aug, DEVICE,
            epochs=epochs, member_train_cfg=spec.get('train'),
        )
        # use the performance-decayed weight (falls back to 1.0 if disabled)
        weight = vote_weight if USE_VAL_WEIGHTS else 1.0
        trained_members.append(dict(
            name=spec['name'], model=model, model_config=model_cfg,
            val_acc=val_acc, val_loss=val_loss, val_recall=val_rec, weight=weight, vote_weight=vote_weight,
            seed=member_seed,
        ))
        member_histories[spec['name']] = history
        if verbose:
            elapsed = time.time() - member_t0
            print(f"  -> final vote weight: {weight:.3f} (val_acc={val_acc:.3f} val_rec={val_rec:.3f}) [{elapsed/60:.1f} min]")

    ensemble = VotingEnsemble(trained_members, vote_mode=VOTE_MODE)
    vote = ensemble.predict(val_x, DEVICE)
    metrics = compute_metrics(val_y, vote['prob'])
    best_single = max(m['val_acc'] for m in trained_members)
    metrics['best_single'] = best_single
    return ensemble, trained_members, member_histories, vote, metrics


def run_stratified_cv(tiles, labels, cv_cfg):
    """Stratified k-fold CV. Each tile is in val exactly once → unbiased OOF metrics."""
    from sklearn.model_selection import StratifiedKFold

    n_splits = cv_cfg['n_splits']
    fold_epochs = cv_cfg.get('epochs_per_fold') or EPOCHS
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=TRAIN_CONFIG['seed'])

    fold_rows = []
    oof_prob = np.zeros(len(labels), dtype=np.float32)
    oof_disagreement = np.zeros(len(labels), dtype=np.float32)

    for fold, (train_idx, val_idx) in enumerate(skf.split(tiles, labels)):
        print(f"\n{'='*20} FOLD {fold + 1}/{n_splits} {'='*20}")
        train_x, train_y = tiles[train_idx], labels[train_idx]
        val_x, val_y = tiles[val_idx], labels[val_idx]

        _, _, _, vote, metrics = train_ensemble_on_split(
            train_x, train_y, val_x, val_y, epochs=fold_epochs, fold_id=fold + 1,
        )
        oof_prob[val_idx] = vote['prob']
        oof_disagreement[val_idx] = vote['disagreement']

        row = dict(fold=fold + 1, n_train=len(train_x), n_val=len(val_y), **metrics)
        fold_rows.append(row)
        print(f"  >> fold {fold + 1}: acc={metrics['acc']:.3f}  recall={metrics['rec']:.3f}  f1={metrics['f1']:.3f}")
        release_gpu()

    oof_metrics = compute_metrics(labels, oof_prob)
    summary = {}
    for key in ['acc', 'prec', 'rec', 'f1', 'best_single']:
        vals = [r[key] for r in fold_rows]
        summary[key] = dict(mean=float(np.mean(vals)), std=float(np.std(vals)), folds=vals)

    return dict(fold_rows=fold_rows, summary=summary, oof_prob=oof_prob,
                oof_disagreement=oof_disagreement, oof_metrics=oof_metrics)

## 8. Train the ensemble

**`RUN_MODE` controls what runs here:**

| Mode | What it does | Time on T4 |
|------|-------------|------------|
| **`deploy`** (default) | Train ensemble once → `ensemble.pt` | ~1–2 hr |
| `evaluate` | 3-fold CV for metrics only | ~3–4 hr |
| `full` | CV + retrain on all data | ~6–8 hr |

For hospitals: **`deploy`** is enough — you get a working model fast.
Run CV later when you have time to report rigorous metrics.

> **If you're stuck in a long CV run:** Runtime → Interrupt, re-run section 2
> with `RUN_MODE = 'deploy'`, then run sections 8a → 8c only.


### 8a. Optional cross-validation (skip if `RUN_MODE = 'deploy'`)


In [ ]:
cv = None
val_prob = val_true = val_x = vote = None
ens_acc = acc = prec = rec = f1 = 0.0
member_histories = {}

if CV_CONFIG['enabled']:
    print(f"Running {CV_CONFIG['n_splits']}-fold stratified CV on {len(tiles)} tiles...")
    cv = run_stratified_cv(tiles, labels, CV_CONFIG)

    print('\n--- CV summary (mean ± std across folds) ---')
    for key in ['acc', 'prec', 'rec', 'f1', 'best_single']:
        s = cv['summary'][key]
        print(f"  {key:<12} {s['mean']:.3f} ± {s['std']:.3f}   folds={[round(v,3) for v in s['folds']]}")

    print('\n--- OOF metrics (unbiased) ---')
    om = cv['oof_metrics']
    print(f"  accuracy {om['acc']:.3f} | precision {om['prec']:.3f} | recall {om['rec']:.3f} | F1 {om['f1']:.3f}")

    val_prob = cv['oof_prob']
    val_true = labels.astype(int)
    val_x = tiles
    vote = dict(prob=cv['oof_prob'], disagreement=cv['oof_disagreement'])
    ens_acc = om['acc']
    acc, prec, rec, f1 = om['acc'], om['prec'], om['rec'], om['f1']
else:
    print(f"Skipping CV (RUN_MODE='{RUN_MODE}'). Training deployment ensemble next.")


### 8b. Train deployment ensemble (this produces `ensemble.pt`)


In [ ]:
ensemble = None
trained_members = []

if CV_CONFIG.get('train_final_on_all', True):
    print(f"\n{'='*20} DEPLOYMENT ENSEMBLE {'='*20}")
    (train_x, train_y), (val_x, val_y) = stratified_split(
        tiles, labels, CV_CONFIG['final_val_fraction'], TRAIN_CONFIG['seed'],
    )
    print(f'train {len(train_x)} | early-stop holdout {len(val_x)}')
    train_t0 = time.time()
    ensemble, trained_members, member_histories, deploy_vote, deploy_metrics = train_ensemble_on_split(
        train_x, train_y, val_x, val_y, fold_id=99,
    )
    print(f"\nDeployment holdout: acc={deploy_metrics['acc']:.3f}  recall={deploy_metrics['rec']:.3f}  f1={deploy_metrics['f1']:.3f}")
    print(f'Total training time: {(time.time()-train_t0)/60:.1f} min')
    if not CV_CONFIG['enabled']:
        val_prob = deploy_vote['prob']
        val_true = val_y.astype(int)
        vote = deploy_vote
        ens_acc = deploy_metrics['acc']
        acc, prec, rec, f1 = deploy_metrics['acc'], deploy_metrics['prec'], deploy_metrics['rec'], deploy_metrics['f1']
    n_members = len(trained_members)
    print('Deployment ensemble ready.')
else:
    n_members = len(ENSEMBLE_MEMBERS)
    print('No deployment training (RUN_MODE=evaluate).')


### 8c. Held-out test evaluation


In [ ]:
test_metrics = None
if EVAL_TEST and ensemble is not None:
    print(f"\n{'='*20} HELD-OUT EVAL ({TEST_SAMPLES} tiles max) {'='*20}")
    if ORGAN_ID == 'lymph_node':
        from datasets import load_dataset
        test_ds = load_dataset('1aurent/PatchCamelyon', split=f'test[:{TEST_SAMPLES}]')
        test_x = np.stack([_resize_tile(img, TILE_SIZE) for img in test_ds['image']])
        test_y = np.array(test_ds['label'], dtype=np.float32)
        print('Using PatchCamelyon official test split')
    else:
        n = min(TEST_SAMPLES, len(val_x))
        test_x, test_y = val_x[:n], val_y[:n]
        print(f'Using validation holdout ({n} tiles) — no official test loader for {ORGAN_ID}')
    test_vote = ensemble.predict(test_x, DEVICE)
    test_metrics = compute_metrics(test_y, test_vote['prob'])
    thr = TRAIN_CONFIG.get('checkpoint_threshold', 0.25)
    prod_metrics = compute_metrics(test_y, test_vote['prob'], threshold=thr)
    print(f"  accuracy  {test_metrics['acc']:.3f}")
    print(f"  precision {test_metrics['prec']:.3f}")
    print(f"  recall    {test_metrics['rec']:.3f}  <- critical for missed cancers")
    print(f"  F1        {test_metrics['f1']:.3f}")
    print(f"  recall @ {thr} (prod threshold): {prod_metrics['rec']:.3f}  |  missed: {prod_metrics['fn']}")
elif ensemble is None:
    print('No ensemble trained — skip test eval.')
else:
    print('Test eval disabled for this RUN_MODE.')


## 8b. Cross-validation results

In [ ]:
if cv is not None:
    folds = [r['fold'] for r in cv['fold_rows']]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, key, title in zip(axes, ['acc', 'rec', 'f1'], ['Accuracy', 'Recall', 'F1']):
        vals = [r[key] for r in cv['fold_rows']]
        ax.bar(folds, vals, color='steelblue', alpha=0.8)
        ax.axhline(cv['summary'][key]['mean'], color='red', ls='--', label=f"mean={cv['summary'][key]['mean']:.3f}")
        ax.set_xlabel('fold'); ax.set_ylim(0, 1); ax.set_title(title); ax.legend(fontsize=8)
    plt.suptitle(f'{CV_CONFIG["n_splits"]}-fold stratified CV — per-fold ensemble metrics'); plt.tight_layout(); plt.show()

    # OOF score distribution
    plt.figure(figsize=(8, 4))
    plt.hist(val_prob[val_true == 0], bins=25, alpha=.6, label='true normal')
    plt.hist(val_prob[val_true == 1], bins=25, alpha=.6, label='true metastasis')
    plt.axvline(0.5, color='k', ls='--'); plt.xlabel('OOF ensemble probability'); plt.legend()
    plt.title('Out-of-fold prediction distribution'); plt.show()
else:
    print('CV disabled — see section 9 for single-split results.')

## 9. Results — ensemble vote (OOF if CV enabled)

In [ ]:
y_pred = (val_prob >= 0.5).astype(int)

if cv is not None:
    print('Metrics from OOF predictions (unbiased CV estimate):')
    print(f"  Ensemble acc={ens_acc:.3f}  recall={rec:.3f}  f1={f1:.3f}")
    print(f"  CV mean acc = {cv['summary']['acc']['mean']:.3f} ± {cv['summary']['acc']['std']:.3f}")
    print(f"  CV mean recall = {cv['summary']['rec']['mean']:.3f} ± {cv['summary']['rec']['std']:.3f}")
    member_accs = {f"fold_avg": cv['summary']['best_single']['mean']}
    if test_metrics is not None:
        print(f"\n  Held-out TEST acc={test_metrics['acc']:.3f}  recall={test_metrics['rec']:.3f}  f1={test_metrics['f1']:.3f}")
else:
    member_accs = {}
    for m in trained_members:
        probs = predict_probs(m['model'], val_x, DEVICE)
        member_accs[m['name']] = ((probs >= 0.5).astype(int) == val_true).mean()
        print(f"  {m['name']:<8} val_acc={member_accs[m['name']]:.3f}  weight={m['weight']:.3f}")
    print(f"\nEnsemble ({VOTE_MODE}) accuracy: {ens_acc:.3f}")

# per-member final vote weights (performance-decayed)
print('\nFinal vote weights (higher = more trusted, fewer mistakes):')
for m in trained_members:
    print(f"  {m['name']:<14} weight={m['weight']:.3f}  val_acc={m['val_acc']:.3f}")

# training curves + weight decay from final deployment model
if member_histories:
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4))
    for name, hist in member_histories.items():
        ep = [h['epoch'] for h in hist]
        a1.plot(ep, [h['val_loss'] for h in hist], 'o-', label=name)
        a2.plot(ep, [h.get('val_rec', h['val_acc']) for h in hist], 'o-', label=name)
        a3.plot(ep, [h.get('vote_weight', 1.0) for h in hist], 'o-', label=name)
    a1.set_title('Val loss per member'); a1.legend(fontsize=8); a1.grid(alpha=.3)
    a2.set_title(f'Val recall @ {TRAIN_CONFIG.get("checkpoint_threshold", 0.25)}'); a2.set_ylim(0, 1); a2.legend(fontsize=8); a2.grid(alpha=.3)
    a3.set_title('Vote weight decay (drops with errors)'); a3.set_ylim(0, 1.05)
    a3.legend(fontsize=8); a3.grid(alpha=.3); a3.set_xlabel('epoch')
    plt.tight_layout(); plt.show()

In [ ]:
tp = int(((y_pred==1)&(val_true==1)).sum())
tn = int(((y_pred==0)&(val_true==0)).sum())
fp = int(((y_pred==1)&(val_true==0)).sum())
fn = int(((y_pred==0)&(val_true==1)).sum())
prec = tp/(tp+fp) if tp+fp else 0
rec  = tp/(tp+fn) if tp+fn else 0
f1   = 2*prec*rec/(prec+rec) if prec+rec else 0
acc  = ens_acc

print(f'Ensemble | Accuracy {acc:.3f} | Precision {prec:.3f} | Recall {rec:.3f} | F1 {f1:.3f}')

cm = np.array([[tn, fp], [fn, tp]])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(['pred neg','pred pos'])
ax.set_yticks([0,1]); ax.set_yticklabels(['true neg','true pos'])
ax.set_title(f'Confusion matrix (ensemble {VOTE_MODE} vote)'); plt.show()

## 9b. Per-tile voting breakdown

Each tile shows how many models voted metastasis. **Disagreement** = models can't agree → flag for pathologist.

In [ ]:
n_members = len(trained_members)
disagree_idx = np.argsort(vote['disagreement'])[::-1][:6]
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
for ax, i in zip(axes, disagree_idx):
    ax.imshow(val_x[i]); ax.axis('off')
    votes = int(vote['votes_for_metastasis'][i])
    ax.set_title(f'{votes}/{n_members} vote meta\np={val_prob[i]:.2f}', fontsize=9)
plt.suptitle('Tiles with most model disagreement'); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(vote['disagreement'][val_true==0], bins=20, alpha=.6, label='true normal')
axes[0].hist(vote['disagreement'][val_true==1], bins=20, alpha=.6, label='true metastasis')
axes[0].set_xlabel('disagreement (std across models)'); axes[0].legend()
axes[1].bar(range(n_members), [member_accs[m['name']] for m in trained_members])
axes[1].set_xticks(range(n_members)); axes[1].set_xticklabels([m['name'] for m in trained_members])
axes[1].set_ylabel('val accuracy'); axes[1].set_title('Per-member accuracy')
plt.tight_layout(); plt.show()

## 10. Monte Carlo Dropout — uncertainty-aware inference

Runs `MC_SAMPLES` forward passes with dropout **on**. Gives a mean probability
and an uncertainty (std). High std = model is unsure → flag for pathologist review.

In [ ]:
def mc_dropout_predict(model, batch_np, device, n_samples=30, batch_size=64):
    """Monte Carlo Dropout on a single member model."""
    model.train()
    norm = get_model_norm(model.cfg)
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
    all_probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            batch_probs = []
            for start in range(0, len(batch_np), batch_size):
                bx = batch_np[start:start+batch_size]
                inp = batch_to_device(tiles_to_tensor(bx, normalize=norm), device)
                with amp_autocast():
                    logits = model(inp)
                batch_probs.append(torch.sigmoid(logits.float()).cpu().numpy())
            all_probs.append(np.concatenate(batch_probs))
    stacked = np.stack(all_probs, axis=0)
    return stacked.mean(axis=0), stacked.std(axis=0)

# MC dropout on best single member + compare to ensemble vote
best_member = max(trained_members, key=lambda m: m['val_acc'])
mc_mean, mc_std = mc_dropout_predict(best_member['model'], val_x, DEVICE, n_samples=MC_SAMPLES)
mc_pred = (mc_mean >= 0.5).astype(int)
mc_acc = (mc_pred == val_true).mean()

print(f'Best member ({best_member["name"]}) accuracy: {best_member["val_acc"]:.3f}')
print(f'Ensemble ({VOTE_MODE}) accuracy:       {ens_acc:.3f}')
print(f'MC Dropout on best member ({MC_SAMPLES} samples): {mc_acc:.3f}')
print(f'Ensemble disagreement (mean):          {vote["disagreement"].mean():.4f}')

uncertain_idx = np.argsort(vote['disagreement'] + mc_std)[::-1][:6]
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, i in zip(axes, uncertain_idx):
    ax.imshow(val_x[i]); ax.axis('off')
    ax.set_title(f'ens={val_prob[i]:.2f}\nmc={mc_mean[i]:.2f}', fontsize=9)
plt.suptitle('Most uncertain tiles (ensemble disagreement + MC dropout)'); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(val_prob, mc_mean, alpha=.3, s=8)
axes[0].plot([0,1],[0,1],'k--'); axes[0].set_xlabel('ensemble prob'); axes[0].set_ylabel('MC dropout prob')
axes[0].set_title('Ensemble vote vs MC Dropout (best member)')
axes[1].hist(mc_std[val_true==0], bins=25, alpha=.6, label='true normal')
axes[1].hist(mc_std[val_true==1], bins=25, alpha=.6, label='true metastasis')
axes[1].set_xlabel('MC std'); axes[1].legend(); axes[1].set_title('MC uncertainty')
plt.tight_layout(); plt.show()

## 11. Triage demo

In [ ]:
demo_tiles = val_x[:64]
demo_vote = ensemble.predict(demo_tiles, DEVICE)
demo_mean, demo_std = mc_dropout_predict(best_member['model'], demo_tiles, DEVICE, n_samples=MC_SAMPLES)
case_score = demo_vote['prob'][np.argsort(demo_vote['prob'])[-10:]].mean()
case_disagreement = demo_vote['disagreement'].mean()
priority = 'URGENT' if case_score >= .85 else ('HIGH' if case_score >= .6 else 'ROUTINE')
if case_disagreement > 0.12:
    priority += ' (models disagree)'
top_idx = np.argsort(demo_vote['prob'])[::-1][:6]

print(f'Case score {case_score:.2f} | disagreement {case_disagreement:.3f} -> {priority}')
for rank, i in enumerate(top_idx, 1):
    v = int(demo_vote['votes_for_metastasis'][i])
    print(f'  {rank}. tile #{i}  ensemble_p={demo_vote["prob"][i]:.2f}  votes={v}/{n_members}')

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for ax, i in zip(axes, top_idx):
    ax.imshow(demo_tiles[i]); ax.axis('off')
    ax.set_title(f'{demo_vote["prob"][i]:.2f}', color='red' if demo_vote['prob'][i]>=.5 else 'green')
plt.suptitle('Suggested review regions (ensemble vote)'); plt.tight_layout(); plt.show()

## 12. Save checkpoint (CPU-loadable)

In [ ]:
checkpoint = {
    'checkpoint_format': 3,
    'organ_id': ORGAN_ID,
    'tile_size': TILE_SIZE,
    'vote_mode': VOTE_MODE,
    'machine_profile': MACHINE_PROFILE,
    'dataset': data_meta,
    'metrics': dict(ensemble_acc=float(ens_acc), f1=float(f1), recall=float(rec), mc_acc=float(mc_acc),
                    test_acc=float(test_metrics['acc']) if test_metrics else None,
                    test_recall=float(test_metrics['rec']) if test_metrics else None),
    'cv_summary': cv['summary'] if cv is not None else None,
    'members': [
        dict(name=m['name'], model_config=m['model_config'], val_acc=m['val_acc'],
             val_recall=m.get('val_recall'), weight=m['weight'], vote_weight=m.get('vote_weight', 1.0),
             seed=m['seed'],
             state_dict={k: v.cpu() for k, v in m['model'].state_dict().items()})
        for m in trained_members
    ],
}
ENSEMBLE_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, ENSEMBLE_PATH)

# Keep the app-ready path and also create a clearly named file for browser download.
# Named copy for browser download (section 13).
if DOWNLOAD_PATH.resolve() != ENSEMBLE_PATH.resolve():
    torch.save(checkpoint, DOWNLOAD_PATH)

print('Saved app checkpoint to:', ENSEMBLE_PATH.resolve())
print('Prepared download file:', DOWNLOAD_PATH.resolve())
if cv is not None:
    print(f'CV OOF recall={rec:.3f} — use this for reporting')

In [ ]:
try:
    from google.colab import files
    files.download(str(DOWNLOAD_PATH))  # one file for this ORGAN_ID only
    print(f'Download started (single checkpoint): {DOWNLOAD_FILENAME}')
    print('Contains all ensemble members for', ORGAN_ID, 'in one .pt file')
except ImportError:
    print('Not on Colab — checkpoint saved at:', ENSEMBLE_PATH.resolve())
    print('Named copy saved at:', DOWNLOAD_PATH.resolve())


## 13. CPU reload sanity check

This section reloads the checkpoint **inside the current Colab runtime** (after section 12).

The browser download in section 13 saves `pathassist_<ORGAN_ID>_ensemble.pt` to your computer. If you use PathAssist locally, copy it to `models/<ORGAN_ID>/ensemble.pt` in that project.


In [ ]:
# CPU reload sanity check inside the current Colab runtime.
# Uses the app path first, then the named download copy.
check_path = None
for candidate in (ENSEMBLE_PATH, DOWNLOAD_PATH):
    if candidate.exists():
        check_path = candidate
        break

if check_path is None:
    raise FileNotFoundError(
        'No checkpoint found in this Colab runtime. Run section 12 first.\n'
        f'Expected one of: {ENSEMBLE_PATH} or {DOWNLOAD_PATH}'
    )

ckpt = torch.load(check_path, map_location='cpu', weights_only=False)
print('Loaded checkpoint:', check_path.resolve())
print('Organ:', ckpt.get('organ_id', ORGAN_ID), '| members:', len(ckpt.get('members', [])))

members = []
for spec in ckpt['members']:
    m = build_model(spec['model_config'])
    m.load_state_dict(spec['state_dict'])
    members.append(dict(name=spec['name'], model=m, weight=spec['weight'], val_acc=spec['val_acc']))

cpu_ensemble = VotingEnsemble(members, vote_mode=ckpt['vote_mode'])
cpu_vote = cpu_ensemble.predict(val_x[:8], 'cpu')
print('CPU ensemble OK | prob:', round(float(cpu_vote['prob'][0]), 3),
      '| votes:', int(cpu_vote['votes_for_metastasis'][0]), '/', len(members))
print(f'Local repo destination after download: models/{ORGAN_ID}/ensemble.pt')

## 14. Quick inference demo (Colab-only)

Score a few validation tiles with the saved ensemble. No external project required.


In [ ]:
check_path = ENSEMBLE_PATH if ENSEMBLE_PATH.exists() else DOWNLOAD_PATH
if not check_path.exists():
    raise FileNotFoundError('Run section 12 first to save the ensemble checkpoint')

ckpt = torch.load(check_path, map_location='cpu', weights_only=False)
demo_members = []
for spec in ckpt['members']:
    m = build_model(spec['model_config'])
    m.load_state_dict(spec['state_dict'])
    demo_members.append(dict(name=spec['name'], model=m, weight=spec['weight'], val_acc=spec['val_acc']))
demo_ensemble = VotingEnsemble(demo_members, vote_mode=ckpt['vote_mode'])

n_demo = min(8, len(val_x))
pred = demo_ensemble.predict(val_x[:n_demo], DEVICE)
for i in range(n_demo):
    truth = 'positive' if val_y[i] >= 0.5 else 'negative'
    call = 'positive' if pred['prob'][i] >= 0.5 else 'negative'
    ok = '✓' if truth == call else '✗'
    print(f"tile {i}: truth={truth:8} pred={call:8} prob={pred['prob'][i]:.3f} {ok}")


In [ ]:
thr = TRAIN_CONFIG.get('checkpoint_threshold', 0.25)
if EVAL_TEST and test_metrics is not None:
    prod = compute_metrics(test_y, test_vote['prob'], threshold=thr)
    print(f"  recall @ {thr} (prod): {prod['rec']:.3f}  |  missed: {prod['fn']}")
elif vote is not None and val_true is not None:
    prod = compute_metrics(val_true, vote['prob'], threshold=thr)
    print(f"  recall @ {thr} (holdout): {prod['rec']:.3f}  |  missed: {prod['fn']}")
else:
    print('Run sections 8b–8c first.')
